# 🚢 Titanic – Exploratory Data Analysis (EDA) Report
### *Who survived the Titanic, and why?*

**Focus areas:** survival overview · **gender vs survival** · passenger class · **age bands** (Infant, Teen, Adult, Old) · family size · embarkation port  
**Charts used:** bar · pie · line · stacked bar · histogram · heatmap

In [ ]:
# ---------- Titanic picture ----------
# Tries to download a real photo of the RMS Titanic (needs internet).
# If offline, it draws a simple illustration of the ship instead.
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Polygon, Rectangle, Circle
import numpy as np, io, urllib.request

IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/f/fd/RMS_Titanic_3.jpg"

def draw_titanic_illustration():
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.set_xlim(0, 22); ax.set_ylim(0, 8); ax.axis('off')
    # sky gradient
    grad = np.linspace(0, 1, 256).reshape(-1, 1)
    ax.imshow(grad, extent=[0, 22, 2, 8], aspect='auto', cmap='Blues_r', alpha=0.55, zorder=0)
    ax.add_patch(Circle((18.5, 6.4), 0.7, color='#ffd966', zorder=1))          # sun
    # sea
    ax.add_patch(Rectangle((0, 0), 22, 2.3, color='#1f4e79', zorder=1))
    # hull
    ax.add_patch(Polygon([(2.5, 2.2), (19.5, 2.2), (20.7, 3.6), (1.8, 3.6)], color='#1a1a1a', zorder=3))
    ax.add_patch(Rectangle((1.8, 3.6), 18.9, 0.25, color='#c0392b', zorder=3))  # red band
    # decks / superstructure
    ax.add_patch(Rectangle((4, 3.85), 13, 0.8, color='#f2f2f2', zorder=3))
    ax.add_patch(Rectangle((5.5, 4.65), 10, 0.6, color='#e6e6e6', zorder=3))
    for i in range(14):                                                           # portholes
        ax.add_patch(Circle((4.4 + i*0.9, 4.2), 0.1, color='#555', zorder=4))
    # funnels
    for x in (6.5, 9.2, 11.9, 14.6):
        ax.add_patch(Rectangle((x, 5.25), 1.1, 1.9, color='#e8a317', zorder=3))
        ax.add_patch(Rectangle((x, 6.8), 1.1, 0.35, color='#111', zorder=4))
    # masts
    ax.plot([3.4, 3.4], [3.85, 6.3], color='#333', lw=2, zorder=3)
    ax.plot([17.8, 17.8], [3.85, 6.0], color='#333', lw=2, zorder=3)
    # waves
    xs = np.linspace(0, 22, 400)
    for k, y0 in enumerate((2.0, 1.4, 0.8)):
        ax.plot(xs, y0 + 0.12*np.sin(xs*2 + k), color='white', alpha=0.5, lw=1.5, zorder=5)
    ax.text(11, 0.3, "RMS TITANIC  •  April 1912", ha='center', color='white',
            fontsize=15, fontweight='bold', zorder=6)
    plt.show()

try:
    req = urllib.request.Request(IMG_URL, headers={"User-Agent": "Mozilla/5.0"})
    img = mpimg.imread(io.BytesIO(urllib.request.urlopen(req, timeout=10).read()), format='jpg')
    plt.figure(figsize=(11, 5)); plt.imshow(img); plt.axis('off')
    plt.title("RMS Titanic (photo: Wikimedia Commons)", fontsize=13); plt.show()
except Exception as e:
    print("Could not download the photo (", type(e).__name__, ") – showing an illustration instead.")
    draw_titanic_illustration()

## 1. Background

The **RMS Titanic** sank on **15 April 1912** after hitting an iceberg on her maiden voyage from Southampton to New York. More than 1,500 people died. There were not enough lifeboats, and the "**women and children first**" rule was followed during the evacuation.

This dataset is the well-known 891-passenger training sample. The goal of this report is to answer:

1. How many passengers survived?
2. **How did gender affect the chance of survival?**
3. Did passenger class, age, family size or port of embarkation matter?
4. Which groups were the most and least likely to survive?

## 2. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

# ---- consistent look & colours for all charts ----
plt.rcParams.update({
    'figure.figsize': (8, 5), 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 14, 'axes.titleweight': 'bold', 'axes.labelsize': 11, 'font.size': 11
})
C_SURV, C_DIED = '#2e8b57', '#c0392b'          # green = survived, red = did not survive
C_F, C_M       = '#e6759b', '#4c78a8'          # female / male
pd.set_option('display.max_columns', 30)

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
if os.path.exists('titanic.csv'):
    df = pd.read_csv('titanic.csv')            # local copy (works offline)
    print("Loaded local titanic.csv")
else:
    df = pd.read_csv(URL)                      # download from GitHub
    print("Loaded from URL")
df.head()

**Data dictionary**

| Column | Meaning |
|---|---|
| `PassengerId` | Unique ID |
| `Survived` | 0 = No, 1 = Yes (**target**) |
| `Pclass` | Ticket class: 1 = upper, 2 = middle, 3 = lower |
| `Name` | Passenger name |
| `Sex` | male / female |
| `Age` | Age in years |
| `SibSp` | # siblings / spouses aboard |
| `Parch` | # parents / children aboard |
| `Ticket` | Ticket number |
| `Fare` | Ticket price (£) |
| `Cabin` | Cabin number |
| `Embarked` | Port: C = Cherbourg, Q = Queenstown, S = Southampton |

## 3. Data Overview & Quality Check

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df.describe(include='object')

### 3.1 Missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'missing': missing, 'percent': pct}))

ax = pct.plot(kind='bar', color=['#e15759', '#f28e2b', '#edc948'], figsize=(7, 4))
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontweight='bold')
ax.set_title("Missing values by column"); ax.set_ylabel("% missing"); ax.set_xlabel("")
plt.xticks(rotation=0); plt.show()

**Observations**
- `Cabin` is ~77% missing → too sparse to use directly (we will create a simple `HasCabin` flag instead).
- `Age` is ~20% missing → we will **impute** it.
- `Embarked` has only 2 missing values → fill with the **mode**.

## 4. Data Cleaning & Feature Engineering

| Step | Method | Why |
|---|---|---|
| Fill `Embarked` | **Mode** | Categorical column |
| Fill `Age` | **Median of the passenger's class + gender group** | Age is skewed; class/gender explain part of the age |
| `HasCabin` | `Cabin.notna()` | Cabin known mostly for upper-class passengers |
| `FamilySize` | `SibSp + Parch + 1` | Total travelling together |
| `Gender` / `Class` labels | readable names for charts | |
| **`AgeBand`** | `pd.cut` | **Infant (0–12), Teen (13–19), Adult (20–59), Old (60+)** |

In [ ]:
data = df.copy()

# 1) Embarked -> mode
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

# 2) Age -> median by (Pclass, Sex)
data['AgeMissing'] = data['Age'].isnull()
data['Age'] = data['Age'].fillna(data.groupby(['Pclass', 'Sex'])['Age'].transform('median'))

# 3) New features
data['HasCabin']   = data['Cabin'].notna()
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
data['Gender']     = data['Sex'].str.capitalize()                         # Female / Male
data['Class']      = data['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})
data['Status']     = data['Survived'].map({1: 'Survived', 0: 'Did not survive'})
data['Port']       = data['Embarked'].map({'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'})

# 4) Age bands
bins   = [0, 12, 19, 59, 120]
labels = ['Infant (0-12)', 'Teen (13-19)', 'Adult (20-59)', 'Old (60+)']
data['AgeBand'] = pd.cut(data['Age'], bins=bins, labels=labels)

print("Remaining nulls (excluding Cabin):")
print(data.drop(columns='Cabin').isnull().sum().sum())
data[['Name', 'Gender', 'Age', 'AgeBand', 'Class', 'FamilySize', 'Status']].head(8)

Quick check of the age bands (how many passengers fall in each):

In [ ]:
data['AgeBand'].value_counts().sort_index()

In [ ]:
# Helper functions to keep chart code short
def pct_labels(ax, fmt="{:.1f}%", fontsize=11, offset=1):
    """Write the value on top of every bar."""
    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width()/2, h + offset),
                        ha='center', va='bottom', fontsize=fontsize, fontweight='bold')

def survival_rate(col):
    """Survival rate (%) per category of a column."""
    return data.groupby(col, observed=True)['Survived'].mean() * 100

overall = data['Survived'].mean() * 100
print(f"Overall survival rate: {overall:.1f}%")

## 5. Overall Survival

### 5.1 How many survived? (Pie chart)

In [ ]:
counts = data['Status'].value_counts()[['Survived', 'Did not survive']]
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(counts, labels=counts.index, colors=[C_SURV, C_DIED], autopct=lambda p: f"{p:.1f}%\n({p*len(data)/100:.0f})",
       startangle=90, explode=(0.05, 0), textprops={'fontsize': 12, 'fontweight': 'bold'},
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title("Overall survival of 891 passengers")
plt.show()

**Insight:** Only about **38%** of the passengers survived – roughly **6 out of 10 passengers died**.

## 6. Passenger Profile (Who was on board?)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Gender
g = data['Gender'].value_counts()
g.plot(kind='bar', color=[C_M, C_F], ax=axes[0]); axes[0].set_title("Passengers by gender")
# Class
c = data['Class'].value_counts().sort_index()
c.plot(kind='bar', color=['#f1c40f', '#95a5a6', '#cd7f32'], ax=axes[1]); axes[1].set_title("Passengers by class")
# Port
p = data['Port'].value_counts()
p.plot(kind='bar', color='#8e7cc3', ax=axes[2]); axes[2].set_title("Passengers by port")

for ax in axes:
    pct_labels(ax, fmt="{:.0f}", offset=5); ax.set_xlabel(""); ax.set_ylabel("Passengers")
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

**Insight:** Men were the majority (~65%), most passengers travelled **3rd class** (~55%), and **Southampton** was by far the busiest boarding port (~72%).

### 6.1 Age distribution (Histogram)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(data['Age'], bins=range(0, 85, 5), color='#7f8c8d', edgecolor='white')
axes[0].axvline(data['Age'].median(), color='black', ls='--', label=f"Median = {data['Age'].median():.0f}")
axes[0].set_title("Age distribution of passengers"); axes[0].set_xlabel("Age"); axes[0].set_ylabel("Passengers")
axes[0].legend()

axes[1].hist([data.loc[data.Survived == 1, 'Age'], data.loc[data.Survived == 0, 'Age']],
             bins=range(0, 85, 5), stacked=True, color=[C_SURV, C_DIED], label=['Survived', 'Did not survive'],
             edgecolor='white')
axes[1].set_title("Age distribution by survival"); axes[1].set_xlabel("Age"); axes[1].legend()
plt.tight_layout(); plt.show()

**Insight:** Most passengers were young adults (20–40). Notice the green share is largest for the youngest ages (0–5).

## 7. Gender vs Survival  ⭐ (Main question)

> **Question: How does gender affect survival rate?**

### 7.1 Survival rate by gender (Bar chart)

In [ ]:
rate_g = survival_rate('Gender')
ax = rate_g.plot(kind='bar', color=[C_F, C_M], figsize=(6, 5), width=0.6, label='_nolegend_')
pct_labels(ax)
ax.axhline(overall, color='gray', ls='--', label=f"Overall average ({overall:.1f}%)")
ax.set_ylim(0, 100); ax.set_ylabel("Survival rate (%)"); ax.set_xlabel("")
ax.set_title("Survival rate by gender"); ax.legend(); plt.xticks(rotation=0); plt.show()

ratio = rate_g['Female'] / rate_g['Male']
print(f"Women were {ratio:.1f}x more likely to survive than men.")

### 7.2 Numbers behind it (Stacked bar – counts)

In [ ]:
ct = pd.crosstab(data['Gender'], data['Status'])[['Survived', 'Did not survive']]
display(ct)

ax = ct.plot(kind='bar', stacked=True, color=[C_SURV, C_DIED], figsize=(6, 5), width=0.6)
for cont in ax.containers:
    ax.bar_label(cont, label_type='center', color='white', fontweight='bold')
ax.set_title("Survivors vs non-survivors by gender (counts)"); ax.set_xlabel(""); ax.set_ylabel("Passengers")
plt.xticks(rotation=0); plt.legend(title=""); plt.show()

Counts can mislead because there were far more men than women. A **100% stacked bar** compares *shares* fairly:

In [ ]:
ct_pct = pd.crosstab(data['Gender'], data['Status'], normalize='index')[['Survived', 'Did not survive']] * 100
ax = ct_pct.plot(kind='barh', stacked=True, color=[C_SURV, C_DIED], figsize=(9, 3.5))
for cont in ax.containers:
    ax.bar_label(cont, fmt='%.1f%%', label_type='center', color='white', fontweight='bold')
ax.set_title("Share who survived vs died, by gender (100% stacked)"); ax.set_ylabel(""); ax.set_xlabel("%")
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, title=""); plt.show()

### 7.3 Pie charts – what happened to women and men?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, gender in zip(axes, ['Female', 'Male']):
    s = data[data.Gender == gender]['Status'].value_counts()[['Survived', 'Did not survive']]
    ax.pie(s, labels=s.index, colors=[C_SURV, C_DIED], autopct='%1.1f%%', startangle=90,
           textprops={'fontsize': 12, 'fontweight': 'bold'}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.set_title(f"{gender} passengers (n={s.sum()})")
plt.tight_layout(); plt.show()

### 7.4 Who made up the survivors? (Pie)

In [ ]:
surv_g = data[data.Survived == 1]['Gender'].value_counts()
plt.figure(figsize=(5.5, 5.5))
plt.pie(surv_g, labels=surv_g.index, colors=[C_F, C_M], autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 12, 'fontweight': 'bold'}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title("Gender split among the 342 survivors"); plt.show()

### 💡 Gender insights
- **Female survival ≈ 74%** vs **male survival ≈ 19%** – women were about **4 times** more likely to survive.
- Although men were ~65% of passengers, they were only ~32% of survivors.
- This matches the historical **"women and children first"** evacuation order.

## 8. Passenger Class vs Survival

In [ ]:
rate_c = survival_rate('Class')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rate_c.plot(kind='bar', color=['#f1c40f', '#95a5a6', '#cd7f32'], ax=axes[0], width=0.6, label='_nolegend_')
pct_labels(axes[0]); axes[0].axhline(overall, color='gray', ls='--', label='Overall average')
axes[0].set_ylim(0, 100); axes[0].set_title("Survival rate by class"); axes[0].set_ylabel("Survival rate (%)")
axes[0].set_xlabel(""); axes[0].legend(); axes[0].tick_params(axis='x', rotation=0)

ct = pd.crosstab(data['Class'], data['Status'])[['Survived', 'Did not survive']]
ct.plot(kind='bar', stacked=True, color=[C_SURV, C_DIED], ax=axes[1], width=0.6)
for cont in axes[1].containers:
    axes[1].bar_label(cont, label_type='center', color='white', fontweight='bold')
axes[1].set_title("Survivors vs non-survivors by class (counts)"); axes[1].set_xlabel("")
axes[1].tick_params(axis='x', rotation=0); axes[1].legend(title="")
plt.tight_layout(); plt.show()

**Insight:** 1st class: ~63% survived · 2nd class: ~47% · 3rd class: ~24%. Money and cabin location (closer to the lifeboats) mattered.

### 8.1 Gender × Class (Grouped bar + Line chart)

In [ ]:
gc = (data.groupby(['Class', 'Gender'])['Survived'].mean().unstack() * 100)[['Female', 'Male']]
display(gc.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# grouped bar
gc.plot(kind='bar', color=[C_F, C_M], ax=axes[0], width=0.7)
for cont in axes[0].containers:
    axes[0].bar_label(cont, fmt='%.0f%%', fontweight='bold', padding=2)
axes[0].set_ylim(0, 115); axes[0].set_title("Survival rate by class and gender"); axes[0].set_ylabel("Survival rate (%)")
axes[0].set_xlabel(""); axes[0].tick_params(axis='x', rotation=0); axes[0].legend(title="")

# line chart
for gender, colr in [('Female', C_F), ('Male', C_M)]:
    axes[1].plot(gc.index, gc[gender], marker='o', lw=3, ms=9, color=colr, label=gender)
    for x, y in zip(gc.index, gc[gender]):
        axes[1].annotate(f"{y:.0f}%", (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontweight='bold')
axes[1].set_ylim(0, 115); axes[1].set_title("Line view: survival drops as class goes down")
axes[1].set_ylabel("Survival rate (%)"); axes[1].set_xlabel("Passenger class"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Insights**
- **Gender mattered in every class**: women had a far higher chance than men everywhere.
- 1st & 2nd class women: **> 90%** survived. 3rd class women: **50%**.
- 3rd-class men had the worst odds: **~14%**.
- Both lines fall from 1st → 3rd class, so **class and gender combine** to shape survival.

## 9. Age Bands vs Survival

Age bands: **Infant (0–12) · Teen (13–19) · Adult (20–59) · Old (60+)**

### 9.1 Survival rate by age band (Bar chart)

In [ ]:
rate_a = survival_rate('AgeBand')
colors_age = ['#f6c85f', '#6f4e7c', '#9fd356', '#ca472f']
ax = rate_a.plot(kind='bar', color=colors_age, figsize=(8, 5), width=0.65, label='_nolegend_')
pct_labels(ax)
ax.axhline(overall, color='gray', ls='--', label=f"Overall average ({overall:.1f}%)")
ax.set_ylim(0, 100); ax.set_ylabel("Survival rate (%)"); ax.set_xlabel("")
ax.set_title("Survival rate by age band"); ax.legend(); plt.xticks(rotation=0); plt.show()
print(data['AgeBand'].value_counts().sort_index().to_string())

### 9.2 Counts per age band (Stacked bar) and passenger share (Pie)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct = pd.crosstab(data['AgeBand'], data['Status'])[['Survived', 'Did not survive']]
ct.plot(kind='bar', stacked=True, color=[C_SURV, C_DIED], ax=axes[0], width=0.65)
for cont in axes[0].containers:
    axes[0].bar_label(cont, label_type='center', color='white', fontweight='bold')
axes[0].set_title("Survivors vs non-survivors by age band"); axes[0].set_xlabel(""); axes[0].set_ylabel("Passengers")
axes[0].tick_params(axis='x', rotation=0); axes[0].legend(title="")

ab = data['AgeBand'].value_counts().sort_index()
axes[1].pie(ab, labels=ab.index, colors=colors_age, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11, 'fontweight': 'bold'}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title("Share of passengers by age band")
plt.tight_layout(); plt.show()

### 9.3 Age band × Gender (Line chart + Grouped bar)

In [ ]:
ag = (data.groupby(['AgeBand', 'Gender'], observed=True)['Survived'].mean().unstack() * 100)[['Female', 'Male']]
n_ag = data.groupby(['AgeBand', 'Gender'], observed=True).size().unstack()
display(ag.round(1)); display(n_ag)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for gender, colr in [('Female', C_F), ('Male', C_M)]:
    axes[0].plot(ag.index.astype(str), ag[gender], marker='o', lw=3, ms=9, color=colr, label=gender)
    for x, y in zip(ag.index.astype(str), ag[gender]):
        axes[0].annotate(f"{y:.0f}%", (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontweight='bold')
axes[0].set_ylim(0, 115); axes[0].set_ylabel("Survival rate (%)"); axes[0].grid(alpha=0.3)
axes[0].set_title("Survival rate across age bands, by gender"); axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

ag.plot(kind='bar', color=[C_F, C_M], ax=axes[1], width=0.7)
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt='%.0f%%', fontweight='bold', padding=2)
axes[1].set_ylim(0, 115); axes[1].set_title("Same data as grouped bars"); axes[1].set_xlabel("")
axes[1].tick_params(axis='x', rotation=15); axes[1].legend(title="")
plt.tight_layout(); plt.show()

**Insights**
- **Infants (0–12)** had the best overall survival (**~58%**), consistent with "children first".
- **Old passengers (60+)** had the lowest survival rate (**~27%**) – but there were only 26 of them, so treat this with care.
- **Adults** are the biggest group (~79% of passengers) and survived at about **37%**.
- Among **infants**, boys and girls survived at similar rates (~57–59%) → age/"children first" mattered more than gender.
- Among **teens and adults**, the gender gap is huge: women ~75–79% vs men ~10–17%.
- ⚠️ Very small groups (e.g. only a handful of women aged 60+) can give extreme values like 100% – interpret carefully.

### 9.4 Survival rate by age, in 10-year steps (Line chart)

In [ ]:
age10 = pd.cut(data['Age'], bins=range(0, 90, 10), right=False)
line = data.groupby(age10, observed=True)['Survived'].agg(['mean', 'count'])
line['mean'] *= 100
x = [f"{int(i.left)}-{int(i.right)-1}" for i in line.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, line['mean'], marker='o', lw=3, ms=9, color='#2c3e50')
ax.fill_between(x, line['mean'], alpha=0.15, color='#2c3e50')
for xi, y, n in zip(x, line['mean'], line['count']):
    ax.annotate(f"{y:.0f}%\n(n={n})", (xi, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
ax.axhline(overall, color='gray', ls='--', label='Overall average')
ax.set_ylim(0, 85); ax.set_xlabel("Age group (years)"); ax.set_ylabel("Survival rate (%)")
ax.set_title("Survival rate drops as age increases"); ax.legend(); ax.grid(alpha=0.3); plt.show()

**Insight:** The 0–9 group has the highest survival rate; after that, survival hovers near 35–40% and falls again for the oldest groups.

### 9.5 Robustness check – only passengers with a *known* age

177 ages were imputed. Do the conclusions hold if we only use real (non-imputed) ages?

In [ ]:
known = data[~data['AgeMissing']]
chk = pd.DataFrame({
    'All passengers (imputed)': survival_rate('AgeBand').round(1),
    'Known ages only': known.groupby('AgeBand', observed=True)['Survived'].mean().mul(100).round(1)
})
chk

The ranking of age bands is the same, so the conclusions are not an artifact of imputation.

## 10. Family Size vs Survival

In [ ]:
fam = data.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
fam['mean'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(fam.index, fam['mean'], marker='o', lw=3, ms=9, color='#8e44ad')
for xi, y in zip(fam.index, fam['mean']):
    axes[0].annotate(f"{y:.0f}%", (xi, y), textcoords='offset points', xytext=(0, 10), ha='center', fontweight='bold')
axes[0].axhline(overall, color='gray', ls='--', label='Overall average')
axes[0].set_xticks(fam.index); axes[0].set_ylim(0, 90); axes[0].set_xlabel("Family size (including self)")
axes[0].set_ylabel("Survival rate (%)"); axes[0].set_title("Survival rate by family size"); axes[0].legend(); axes[0].grid(alpha=0.3)

fam_group = pd.cut(data['FamilySize'], [0, 1, 4, 20], labels=['Alone (1)', 'Small (2-4)', 'Large (5+)'])
r = data.groupby(fam_group, observed=True)['Survived'].mean() * 100
r.plot(kind='bar', color=['#e67e22', '#27ae60', '#c0392b'], ax=axes[1], width=0.6)
pct_labels(axes[1]); axes[1].set_ylim(0, 100); axes[1].set_xlabel(""); axes[1].set_ylabel("Survival rate (%)")
axes[1].set_title("Travelling alone vs small vs large family"); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

**Insight:** Passengers travelling in **small families (2–4)** survived most (~55–72%). People **alone (~30%)** and in **large families (5+ , ~16%)** did worse – large families were mostly in 3rd class.

## 11. Port of Embarkation & Cabin

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

r = survival_rate('Port')
r.plot(kind='bar', color='#8e7cc3', ax=axes[0], width=0.6); pct_labels(axes[0])
axes[0].set_ylim(0, 100); axes[0].set_title("Survival rate by port"); axes[0].set_xlabel(""); axes[0].set_ylabel("Survival rate (%)")
axes[0].tick_params(axis='x', rotation=0)

pc = pd.crosstab(data['Port'], data['Class'], normalize='index') * 100
pc.plot(kind='bar', stacked=True, color=['#f1c40f', '#95a5a6', '#cd7f32'], ax=axes[1], width=0.6)
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt='%.0f%%', label_type='center', fontsize=9, fontweight='bold')
axes[1].set_title("Class mix at each port (stacked)"); axes[1].set_xlabel(""); axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title="Class", fontsize=9)

r2 = survival_rate('HasCabin').rename({True: 'Cabin recorded', False: 'No cabin info'})
r2.plot(kind='bar', color=['#7f8c8d', '#16a085'], ax=axes[2], width=0.6); pct_labels(axes[2])
axes[2].set_ylim(0, 100); axes[2].set_title("Survival by cabin information"); axes[2].set_xlabel("")
axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

**Insights**
- **Cherbourg** passengers survived most (~55%) – a larger share of them were **1st class**. The port itself is not the cause; the class mix explains most of the difference.
- Passengers with a **recorded cabin** (mostly 1st class) survived at ~67% vs ~30% without.

## 12. Fare vs Survival

In [ ]:
data['FareBand'] = pd.qcut(data['Fare'], 4, labels=['Q1 (cheapest)', 'Q2', 'Q3', 'Q4 (most expensive)'])
r = survival_rate('FareBand')
ax = r.plot(kind='bar', color=['#d6eaf8', '#85c1e9', '#3498db', '#1b4f72'], figsize=(8, 5), width=0.65)
pct_labels(ax); ax.set_ylim(0, 100); ax.set_xlabel("Fare quartile"); ax.set_ylabel("Survival rate (%)")
ax.set_title("Higher fare, higher survival"); plt.xticks(rotation=0); plt.show()
print("Median fare – survivors: £%.1f | non-survivors: £%.1f" % (
    data.loc[data.Survived == 1, 'Fare'].median(), data.loc[data.Survived == 0, 'Fare'].median()))

## 13. Heatmap – Combining Gender, Class and Age Band

In [ ]:
pivot = data.pivot_table(values='Survived', index=['Gender', 'Class'], columns='AgeBand', aggfunc='mean', observed=True) * 100
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns, rotation=15)
ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels([f"{g} - {c}" for g, c in pivot.index])
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, "n/a" if np.isnan(v) else f"{v:.0f}%", ha='center', va='center', fontweight='bold')
plt.colorbar(im, label="Survival rate (%)"); ax.set_title("Survival rate (%) by gender, class and age band")
plt.tight_layout(); plt.show()

**Insight:** The green zone is concentrated in **women in 1st/2nd class** and **infants**; the red zone is **men aged 13+, especially in 3rd class**. `n/a` means no passengers in that group.

## 14. Correlation with Survival

In [ ]:
num = data[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']].copy()
num['Female'] = (data['Sex'] == 'female').astype(int)
corr = num.corr()['Survived'].drop('Survived').sort_values()
ax = corr.plot(kind='barh', color=[C_DIED if v < 0 else C_SURV for v in corr], figsize=(7, 4.5))
for p in ax.patches:
    w = p.get_width(); ax.annotate(f"{w:+.2f}", (w, p.get_y() + p.get_height()/2),
                                   ha='left' if w > 0 else 'right', va='center', fontweight='bold')
ax.axvline(0, color='black', lw=0.8); ax.set_xlim(-0.6, 0.7)
ax.set_title("Correlation of each feature with Survived"); ax.set_xlabel("Pearson correlation"); plt.show()

**Insight:** Being **female** has the strongest positive link to survival; **higher class number (3rd class)** has the strongest negative link.

## 15. Questions (Survival-based)

Answer these using `data` (the cleaned DataFrame). Try first, then compare with Section 16.

**Basic**
1. How many passengers survived and how many died? What is the overall survival rate?
2. How many male and female passengers were there?
3. What is the survival rate for men and for women? How many times higher is the women's rate?
4. What percentage of all survivors were women?

**Class & Gender**
5. Which passenger class had the highest and lowest survival rate?
6. Create a table of survival rate by **Class × Gender**. Which group had the highest rate and which the lowest?
7. Which class had the highest number of deaths in absolute terms?

**Age bands**
8. How many passengers are in each age band (Infant, Teen, Adult, Old)?
9. Which age band had the highest survival rate? Which had the lowest?
10. Within the **Adult** band, compare male vs female survival rate.
11. What is the average age of survivors vs non-survivors?
12. Among *children under 10*, what percentage survived?

**Family & others**
13. Did passengers travelling alone survive more or less than those with family?
14. Which port of embarkation had the highest survival rate and why might that be?
15. What is the median fare paid by survivors vs non-survivors?

**Challenge**
16. Plot a **stacked bar** of survival counts by `AgeBand` for **women only**.
17. Find the survival rate of **women in 3rd class who embarked at Southampton**.
18. Build a function `survival_by(col)` that returns survival rate (%) and passenger count for any column and test it on `Class`, `Gender`, `AgeBand`.
19. If you had to predict survival with one simple rule (`female → survived, male → died`), what accuracy would it give?
20. Write a 3-line summary of the main factors that influenced survival.

## 16. Solutions

In [ ]:
# Q1
print(data['Status'].value_counts()); print(f"Survival rate: {data['Survived'].mean():.1%}")
# Q2
print(data['Gender'].value_counts())
# Q3
r = data.groupby('Gender')['Survived'].mean(); print(r.round(3)); print("Ratio F/M:", round(r['Female']/r['Male'], 2))
# Q4
print(f"{(data[data.Survived==1]['Gender'].value_counts(normalize=True)['Female']):.1%} of survivors were women")

In [ ]:
# Q5
r = data.groupby('Class')['Survived'].mean().sort_values(ascending=False); print(r.round(3))
print("Highest:", r.index[0], "| Lowest:", r.index[-1])
# Q6
t = data.pivot_table(values='Survived', index='Class', columns='Gender', aggfunc='mean').round(3); display(t)
print("Highest:", t.stack().idxmax(), "| Lowest:", t.stack().idxmin())
# Q7
print(data[data.Survived == 0]['Class'].value_counts())

In [ ]:
# Q8
print(data['AgeBand'].value_counts().sort_index())
# Q9
r = data.groupby('AgeBand', observed=True)['Survived'].mean(); print(r.round(3)); print("Highest:", r.idxmax(), "| Lowest:", r.idxmin())
# Q10
print(data[data.AgeBand == 'Adult (20-59)'].groupby('Gender')['Survived'].mean().round(3))
# Q11
print(data.groupby('Status')['Age'].mean().round(1))
# Q12
kids = data[data.Age < 10]; print(f"{kids['Survived'].mean():.1%} of {len(kids)} children under 10 survived")

In [ ]:
# Q13
alone = data['FamilySize'] == 1
print("Alone     :", round(data[alone]['Survived'].mean(), 3))
print("With family:", round(data[~alone]['Survived'].mean(), 3))
# Q14
r = data.groupby('Port')['Survived'].mean().sort_values(ascending=False); print(r.round(3))
print("-> Cherbourg is highest; it had a much higher share of 1st-class passengers.")
print(pd.crosstab(data['Port'], data['Class'], normalize='index').round(2))
# Q15
print(data.groupby('Status')['Fare'].median())

In [ ]:
# Q16
w = data[data.Gender == 'Female']
ct = pd.crosstab(w['AgeBand'], w['Status'])[['Survived', 'Did not survive']]
ax = ct.plot(kind='bar', stacked=True, color=[C_SURV, C_DIED], figsize=(7, 4.5))
for cont in ax.containers: ax.bar_label(cont, label_type='center', color='white', fontweight='bold')
ax.set_title("Women: survival by age band"); ax.set_xlabel(""); plt.xticks(rotation=0); plt.show()
# Q17
q = data[(data.Gender == 'Female') & (data.Class == '3rd') & (data.Embarked == 'S')]
print(f"{q['Survived'].mean():.1%} (n={len(q)})")

In [ ]:
# Q18
def survival_by(col):
    out = data.groupby(col, observed=True)['Survived'].agg(['mean', 'count'])
    out['mean'] = (out['mean'] * 100).round(1)
    return out.rename(columns={'mean': 'survival_%', 'count': 'passengers'})
for c in ['Class', 'Gender', 'AgeBand']:
    display(survival_by(c))
# Q19
pred = (data['Sex'] == 'female').astype(int)
print(f"Accuracy of the simple gender rule: {(pred == data['Survived']).mean():.1%}")

**Q20 (sample answer):** Survival was strongly linked to **gender** (women ≈ 74% vs men ≈ 19%), **passenger class** (1st ≈ 63% → 3rd ≈ 24%) and **age** (infants highest, 60+ lowest). The effects combine: 1st/2nd-class women almost all survived, while 3rd-class men rarely did.

## 17. Key Insights – Summary

| # | Insight | Evidence |
|---|---|---|
| 1 | **Only ~38% survived** | 342 of 891 passengers |
| 2 | **Gender was the strongest factor** | Women 74% vs men 19% (≈ 4× higher) |
| 3 | Men were 65% of passengers but only ~32% of survivors | "Women and children first" |
| 4 | **Class mattered** | 1st: 63% · 2nd: 47% · 3rd: 24% |
| 5 | **Gender and class combine** | 1st/2nd-class women > 90% · 3rd-class men ≈ 14% |
| 6 | **Infants had the best odds** | ~58% survival; boys and girls similar |
| 7 | **Teen and adult men fared worst** | Only ~10–17% survived |
| 8 | Oldest group (60+) had the lowest age-band survival | ~27% (small sample, n = 26) |
| 9 | **Small families did best** | 2–4 people: ~55–72%; alone ~30%; 5+ ~16% |
| 10 | Port & cabin effects are mostly **class effects in disguise** | Cherbourg had more 1st class passengers |
| 11 | **Higher fare → higher survival** | Median fare £26 (survived) vs £10.5 (did not) |

### Limitations
- Only **891 of ~2,224** people on board are in this dataset.
- **Age** had 177 imputed values; very small groups (e.g. women 60+) give unstable percentages.
- This analysis shows **association, not causation**.

## 18. Final Conclusion

The Titanic disaster was **not random**. Survival depended heavily on **who you were**:

1. 🚺 **Gender was the single biggest factor.** A woman was about **four times** as likely to survive as a man (74% vs 19%).
2. 🎟️ **Class created a second divide.** Survival fell steadily from 1st (63%) to 3rd class (24%), and the gender gap existed in all classes.
3. 👶 **Age mattered, especially for the youngest.** Infants (0–12) had the best overall survival (~58%), while teen and adult **men** had the lowest.
4. 👪 **Travelling with a small family helped**, whereas being alone or in a very large family lowered the odds.
5. 💰 Fare, cabin and port differences are largely **side-effects of passenger class**.

**Overall:** the safest profile was a **woman or young child in 1st or 2nd class**; the most at-risk profile was an **adult man in 3rd class**. This reflects both the "women and children first" policy and the social inequality of 1912.

**Next steps:** build a machine-learning model (Logistic Regression / Random Forest) using `Sex`, `Pclass`, `Age`, `FamilySize`, `Fare` to predict survival; extract passenger **titles** (Mr, Mrs, Miss, Master) from `Name` as another feature.

In [ ]:
# ---------- One-page dashboard ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Titanic survival – at a glance", fontsize=18, fontweight='bold')

rate_g.plot(kind='bar', color=[C_F, C_M], ax=axes[0, 0], width=0.55); pct_labels(axes[0, 0])
axes[0, 0].set_title("By gender"); axes[0, 0].set_ylim(0, 100)

rate_c.plot(kind='bar', color=['#f1c40f', '#95a5a6', '#cd7f32'], ax=axes[0, 1], width=0.55); pct_labels(axes[0, 1])
axes[0, 1].set_title("By class"); axes[0, 1].set_ylim(0, 100)

rate_a.plot(kind='bar', color=colors_age, ax=axes[1, 0], width=0.55); pct_labels(axes[1, 0])
axes[1, 0].set_title("By age band"); axes[1, 0].set_ylim(0, 100)

for gender, colr in [('Female', C_F), ('Male', C_M)]:
    axes[1, 1].plot(gc.index, gc[gender], marker='o', lw=3, ms=9, color=colr, label=gender)
axes[1, 1].set_title("Class × gender"); axes[1, 1].set_ylim(0, 105); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

for ax in axes.ravel():
    ax.set_xlabel(""); ax.set_ylabel("Survival rate (%)"); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()